# MPC on M3, compared with the MPCs on M2 and F

The same delay-compensated MPC as on M2 (`softarm.controllers.DelayMPC`), built on **M3** instead: M2's
actuator with **second-order** mechanics (a lumped mass per axis). In the model comparison M3 was the most
accurate free-run model. Here it is used inside the controller, on `make_calibrated_sim(control_hz=100)`
(0.5 s dead time, 0.6 s lag), with the reference known in advance, on the six comparison tests.

| Controller | Model | Horizon | Weight on command changes |
|---|---|---|---|
| MPC (M2) | first-order mechanics, state $(y, p)\in\mathbb R^6$ | 100 steps = 1.00 s | $R_u = 10^{-5}$ (tuned) |
| **MPC (M3)** | second-order mechanics, state $(y, \dot y, p)\in\mathbb R^9$ | 100 steps = 1.00 s | $R_u$ tuned below |
| MPC (F) | frequency-identified, per channel, Kalman filter | 300 steps = 3.00 s | $\rho = 10^{-5}$ m²/psi² |

The controller (`softarm.controllers.DelayMPC` on `softarm.models.M3`), tests and metrics are in the `softarm`
package; `scripts/mpc_m3.py` (R_u tuning, tracking figure) and `scripts/mpc_signals.py` (runs, logging, figures)
run the experiment. This notebook calls them and shows the results. The derivation of the MPC law is on the MPC page
(`docs/reports/mpc-on-m2-and-f.html`). Runs offline in the project environment (`uv run jupyter lab`), about 30 s.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))   # the experiment scripts (tuning, runs, figures)

import numpy as np
from IPython.display import Markdown, display
from softarm import plotting as P
from softarm.constants import HZ
from softarm.controllers import DelayMPC
from softarm.evaluation import TESTS, TUNE, circle, ref_array, track_ref_mm
from softarm.models import M3
import mpc_m3
import mpc_signals as S

%matplotlib inline
P.apply()
NAMES = (S.M2, S.M3, S.F)


def md_table(head, rows):
    out = ["| " + " | ".join(head) + " |", "|---" * len(head) + "|"]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

## 1 — The M3 model

Per axis $i \in \{x, y, z\}$, with $y$ the tip (mm) and $p$ the measured S2–S4 pressure (psi), both relative to the settled pose:

$$\tau\,\dot p = -p + G\,u(t-T), \qquad \ddot y_i + 2\zeta_i\omega_i\,\dot y_i + \omega_i^2\,y_i = \omega_i^2\,(Hp)_i$$

State-space form, $X = (y, \dot y, p) \in \mathbb R^9$:

$$\dot X = \begin{bmatrix} 0 & I & 0 \\ -\Omega^2 & -2Z\Omega & \Omega^2 H \\ 0 & 0 & -\tfrac1\tau I \end{bmatrix} X
 + \begin{bmatrix} 0 \\ 0 \\ \tfrac1\tau \operatorname{diag}(G) \end{bmatrix} u(t-T), \qquad y = [\,I\ \ 0\ \ 0\,]\,X$$

M2 is the first-order limit of this, $\dot y = -\Lambda y + \Lambda H p$. The controller measures $y$ (tip) and $p$
(pouch sensors); $\dot y$ is the backward difference of the measured tip, a signal the hardware also has.
Parameters from `docs/figures/task_space_model.json` (`M3_mechanics`), fitted by `scripts/task_space_model.py`:

In [ ]:
prm = M3.load().params()
md_table(["Axis", "ω (rad/s)", "ζ", "H row (mm/psi, S2 S3 S4)"],
         [[a, f"{prm['wn'][i]:.2f}", f"{prm['zeta'][i]:.3f}", ", ".join(f"{h:+.3f}" for h in prm["H"][i])] for i, a in enumerate("xyz")])
print(f"τ = {prm['tau_s']:.3f} s, G = {np.round(prm['gain'], 3)}, T = {prm['delay_s']} s")

## 2 — The MPC law (unchanged from M2)

Every 10 ms: predict across the dead time with the commands already sent, add the offset $w$ (tip prediction
error, low-passed 0.5 s), and minimise over the next $H$ commands

$$J = \sum_{i=0}^{H-1} \big\| C\hat X_{k+N+1+i} + w - r_{k+N+i} \big\|^2 + R_u \sum_{i=0}^{H-1} \big\| u_{k+i} - u_{k+i-1} \big\|^2, \qquad N = 50$$

in closed form; apply the first move $u_k = g_r\,\mathcal R + g_x\,\hat X_{k+N} + g_w\,w + g_u\,u_{k-1}$, clipped to 0–9 psi.
Only $A$, $B$, $C$ change between M2 and M3.

## 3 — Tuning $R_u$

Same grid and tuning task as the M2 MPC (5 mm, 0.07 Hz circle, sim seed 5; never a test). RMSE after the first lap, mm:

In [ ]:
ru_m3, grid = mpc_m3.tune()
md_table(["R_u (mm²/psi²)"] + [f"{r:g}" for r, _ in grid], [["MPC (M3), 100 steps"] + [f"{e:.4f}" for _, e in grid]])
print(f"chosen R_u = {ru_m3:g}   (MPC on M2 at the same horizon: 0.0148 mm at R_u = 1e-05)")
assert ru_m3 == S.M3_RU, "mpc_signals.M3_RU must match the tuned value"


On the tuning circle the M3 MPC is worse than the M2 MPC (0.037 against 0.015 mm). The next cell checks whether
the differenced velocity is the cause, by giving the controller the simulator's own tip velocity (diagnostic only:
not available on hardware) or by setting $\dot y = 0$.

In [ ]:
d_tune = ref_array(circle(TUNE["radius_m"], TUNE["frequency_hz"]))


def tune_score(ctl):
    res = track_ref_mm(ctl, d_tune, 3000, seed=TUNE["seed"])
    e = 1000 * np.linalg.norm((res["tips"] - res["refs"])[:, :2], axis=1)
    return np.sqrt(np.mean(e[int(round(HZ / TUNE["frequency_hz"])):] ** 2))


def with_velocity(vel):
    """The M3 MPC with y' in its state taken from vel(obs) (mm/s) instead of the differenced tip."""
    m = M3.load()
    ctl = DelayMPC(m, d_tune, mpc_m3.HORIZON, ru_m3)
    seen = []
    m.measured_state = lambda y, y_prev, p: np.concatenate([y, vel(seen[-1]), p])
    ctl.compute = lambda t, obs, ref: seen.append(obs) or DelayMPC.compute(ctl, t, obs, ref)
    return ctl


sim_vel = lambda obs: 1000 * np.asarray(obs["tip_vel"])
zero_vel = lambda obs: np.zeros(3)
md_table(["Velocity used in the M3 state", "RMSE after first lap (mm)"],
         [["differenced measured tip (used)", f"{tune_score(DelayMPC(M3.load(), d_tune, mpc_m3.HORIZON, ru_m3)):.4f}"],
          ["simulator tip_vel [diagnostic]", f"{tune_score(with_velocity(sim_vel)):.4f}"],
          ["ẏ = 0 [diagnostic]", f"{tune_score(with_velocity(zero_vel)):.4f}"]])

## 4 — Run the three MPCs on the six tests

Same tests and metrics as the M2 controller comparison: planar x–y error in mm; *after settling* = after the first
lap (circles, triangle, square) or the first 5 s (random paths); *shape error* = RMS distance to the nearest point of
the reference path, ignoring timing; *saturation* = % of commands at 0 or 9 psi.

In [ ]:
rows = S.run_all(NAMES)
R = {(r["test"], r["controller"]): r["metrics"] for r in rows}
md_table(["Test"] + [P.label(n) for n in NAMES],
         [[t] + [f"{R[(t, n)]['rmse_settled_mm']:.3f}" for n in NAMES] for t in TESTS])
display(Markdown("RMSE after settling (mm). Saturation (%):"))
md_table(["Test"] + [P.label(n) for n in NAMES],
         [[t] + [f"{R[(t, n)]['saturated_pct']:.1f}" for n in NAMES] for t in TESTS])


def results(test):
    kind = TESTS[test][3]
    after = "after 5 s" if kind == "open" else "after first lap"
    head = ["Controller", "RMSE whole run", f"RMSE {after}", "Shape error", f"Max {after}", "Saturation %"]
    head += {"circle": ["Radius range"], "polygon": ["Path size %"], "open": []}[kind]
    out = []
    for n in NAMES:
        m = R[(test, n)]
        row = [n, f"{m['rmse_whole_mm']:.2f}", f"{m['rmse_settled_mm']:.3f}", f"{m['shape_mm']:.3f}",
               f"{m['max_settled_mm']:.2f}", f"{m['saturated_pct']:.1f}"]
        if kind == "circle":
            row.append(f"{m['radius_min_mm']:.1f}–{m['radius_max_mm']:.1f} (ref {m['radius_ref_mm']:.1f})")
        if kind == "polygon":
            row.append(f"{m['size_pct']:.0f}")
        out.append(row)
    md_table(head, out)


def show(test):
    results(test)
    mpc_m3.tracking_figure(rows, test, NAMES)
    for n in NAMES:
        S.figure_one(rows, test, n)
    S.phase(rows, test, names=NAMES)

## 5 — Results per test

For each test: the metrics table; the tip path and error over time; pressures and disturbance (commanded and measured
pressure, the actual disturbance $d = \dot x - (-a x + bH(p - p_0))$ measured against M2, and each MPC's own offset
estimate); and the error-space phase portrait ($E$ = predicted tip at arrival + offset − reference then, against $\dot E$).

### held-out circle

In [ ]:
show("held-out circle")

### scored circle

In [ ]:
show("scored circle")

### triangle

In [ ]:
show("triangle")

### square

In [ ]:
show("square")

### squiggle

In [ ]:
show("squiggle")

### random waypoints

In [ ]:
show("random waypoints")

## 6 — Findings (from the run above)

- **M3 beats M2 inside the MPC on every path with corners or random turns**, at the same horizon and tuning:
  triangle 0.069 vs 0.129 mm, square 0.069 vs 0.124, waypoints 0.105 vs 0.167, squiggle 0.041 vs 0.071 (RMSE after settling).
  On the squiggle it is the best of the three; on the triangle and square it is within 0.003 mm of the MPC on F (0.066, 0.068).
- **It is worse on the scored circle** (0.074 vs 0.029 mm for M2) and tied on the held-out circle (reach-limited for all).
- **It saturates far more**: 6–46 % of commands at 0 or 9 psi, against 1–23 % for M2 and 0–19 % for F. With
  $R_u = 10^{-5}$ it nearly inverts a second-order model, which asks for sharper pressure changes. Before using it on the
  robot, check a larger $R_u$ or a longer horizon on a random path (different seed), since the tuning circle has no corners.
- **The velocity estimate is not the limitation**: the simulator's own tip velocity gives the same tuning score; setting
  $\dot y = 0$ is slightly better, which points at the second-order dynamics themselves on the smooth circle.
- **Caveat (added 2026-11-10):** these results use path seed 101 only. On other seeds the M3 MPC can be much worse: squiggle seed 103 0.302 mm with 66 % of commands saturated (M2: 0.029 mm); waypoints seed 102 0.281 mm, 51 % saturated (M2: 0.231 mm). Measured in `Soft_robotic_arm_MPC_M3.ipynb`.